# 04 — Data Preparation

**Objective**: Clean, analyze and prepare the evaluation database for surrogate training.

- Load `eval_database_v2.json` (60k samples with consistent c_ref + control derivatives)
- Filter divergent AVL outputs
- Feature engineering (30 raw → 52 augmented)
- Distribution analysis of 10 surrogate targets

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.parameterization.design_variables import VAR_NAMES, BOUNDS, N_VARS
from src.surrogate.model import PRIMITIVE_TARGETS, _TARGET_CLIP
from src.surrogate.features import augment_features
from src.optimization.database import EvaluationDatabase

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams.update({"figure.dpi": 130, "figure.facecolor": "white"})

In [ ]:
%load_ext autoreload
%autoreload 2

db = EvaluationDatabase.load('../data/eval_database_v2.json')
X_arr, results = db.to_arrays()
print(f'Loaded {len(db)} evaluations')
print(f'X shape: {X_arr.shape}')

# Check v2 fields
n_ctrl = sum(1 for r in results if 'Cmd01' in r)
n_cref = sum(1 for r in results if 'c_ref' in r)
print(f'With control derivatives: {n_ctrl}/{len(results)}')
print(f'With explicit c_ref: {n_cref}/{len(results)}')

In [ ]:
# ── Chargement ──
db = EvaluationDatabase.load('../data/eval_database.json')
X_raw, results = db.to_arrays()

print(f"Nombre total d'évaluations : {len(results)}")
print(f"Dimension X : {X_raw.shape}")
print(f"Clés d'un résultat : {sorted(results[0].keys())}")

# DataFrame des variables de design
df_X = pd.DataFrame(X_raw, columns=VAR_NAMES)
print(f"\n── Variables de design ──")
df_X.describe().round(4)

In [ ]:
# ── Extraction des 7 primitives VLM (même logique que model.py) ──
rows = []
for r in results:
    alpha_eq_rad = np.radians(r.get("alpha_eq", 0.0))
    cl_alpha = r.get("CL_alpha", 0.0)
    cm_alpha = r.get("CM_alpha", 0.0)

    cl_0 = r.get("CL_0", None)
    if cl_0 is None:
        cl_0 = r.get("CL_required", 0.0) - cl_alpha * alpha_eq_rad
    cm_0 = r.get("CM_0", None)
    if cm_0 is None:
        cm_0 = r.get("CM", 0.0) - cm_alpha * alpha_eq_rad

    cd0_wing = r.get("CD0_wing", r.get("CD0", 0.008) * 0.6)
    cd0_body = r.get("CD0_body", r.get("CD0", 0.008) * 0.4)
    cn_beta = r.get("Cn_beta", 0.0)

    rows.append([cl_0, cl_alpha, cm_0, cm_alpha, cd0_wing, cd0_body, cn_beta])

df_Y = pd.DataFrame(rows, columns=PRIMITIVE_TARGETS)
print(f"Shape targets : {df_Y.shape}")
df_Y.describe().round(6)

## 2. Détection des outliers VLM

Le solveur VLM (Vortex Lattice Method) diverge numériquement pour certaines géométries extrêmes.
Ces divergences produisent des valeurs aberrantes (ex. CL_0 = 1e10) qui polluent l'entraînement.

On utilise les bornes physiques `_TARGET_CLIP` définies dans `model.py` comme référence,
puis on affine l'analyse.

In [ ]:
# ── Tableau de diagnostic par primitive ──
print("Bornes physiques (_TARGET_CLIP) :")
print(f"{'Primitive':<12} {'lo':>10} {'hi':>10}")
print("-" * 34)
for key in PRIMITIVE_TARGETS:
    lo, hi = _TARGET_CLIP[key]
    print(f"{key:<12} {lo:>10.3f} {hi:>10.3f}")

print("\n── Analyse des outliers par cible ──\n")
print(f"{'Cible':<12} {'N total':>8} {'In bounds':>10} {'Outliers':>10} {'% outliers':>12} "
      f"{'Min':>14} {'Max':>14} {'P1':>10} {'P99':>10}")
print("-" * 110)

outlier_masks = {}
for key in PRIMITIVE_TARGETS:
    vals = df_Y[key].values
    lo, hi = _TARGET_CLIP[key]
    in_bounds = (vals >= lo) & (vals <= hi)
    n_in = in_bounds.sum()
    n_out = len(vals) - n_in
    pct = 100 * n_out / len(vals)
    p1, p99 = np.percentile(vals, [1, 99])
    outlier_masks[key] = ~in_bounds
    print(f"{key:<12} {len(vals):>8} {n_in:>10} {n_out:>10} {pct:>11.1f}% "
          f"{vals.min():>14.4f} {vals.max():>14.4f} {p1:>10.4f} {p99:>10.4f}")

In [ ]:
# ── Histogrammes : distribution brute vs bornes physiques ──
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flat

for i, key in enumerate(PRIMITIVE_TARGETS):
    ax = axes[i]
    vals = df_Y[key].values
    lo, hi = _TARGET_CLIP[key]

    # Clamp pour la visualisation (sinon les outliers écrasent l'histogramme)
    vals_clamp = np.clip(vals, lo - 0.5 * (hi - lo), hi + 0.5 * (hi - lo))

    ax.hist(vals_clamp, bins=80, alpha=0.7, edgecolor="none", color="steelblue")
    ax.axvline(lo, color="red", ls="--", lw=1.5, label=f"lo = {lo}")
    ax.axvline(hi, color="red", ls="--", lw=1.5, label=f"hi = {hi}")

    n_out = outlier_masks[key].sum()
    pct = 100 * n_out / len(vals)
    ax.set_title(f"{key}\n{n_out} outliers ({pct:.1f}%)", fontweight="bold", fontsize=10)
    ax.legend(fontsize=7)
    ax.set_ylabel("Count")

axes[7].set_visible(False)
fig.suptitle("Distribution des primitives VLM — bornes physiques en rouge", fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Matrice de co-occurrence des outliers ──
# Un sample peut être outlier sur plusieurs primitives à la fois
outlier_df = pd.DataFrame(outlier_masks)
outlier_any = outlier_df.any(axis=1)
print(f"Samples outlier sur au moins 1 primitive : {outlier_any.sum()} / {len(outlier_any)} "
      f"({100*outlier_any.mean():.1f}%)")

# Nombre de primitives en outlier par sample
n_outlier_per_sample = outlier_df.sum(axis=1)
print(f"\nDistribution du nombre de primitives en outlier par sample :")
for n in range(8):
    count = (n_outlier_per_sample == n).sum()
    if count > 0:
        print(f"  {n} primitives en outlier : {count} samples ({100*count/len(n_outlier_per_sample):.1f}%)")

# Matrice de co-occurrence
cooccurrence = outlier_df.astype(int).T @ outlier_df.astype(int)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cooccurrence, annot=True, fmt="d", cmap="YlOrRd", ax=ax,
            xticklabels=PRIMITIVE_TARGETS, yticklabels=PRIMITIVE_TARGETS)
ax.set_title("Co-occurrence des outliers entre primitives", fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Caractérisation des géométries divergentes

Quelles variables de design provoquent la divergence VLM ?
On compare la distribution des variables entre samples propres et outliers.

In [ ]:
# ── Test statistique : quelles variables discriminent clean vs outlier ? ──
from scipy.stats import ks_2samp

mask_clean = ~outlier_any.values
mask_outlier = outlier_any.values

print(f"Clean: {mask_clean.sum()} | Outlier: {mask_outlier.sum()}\n")
print(f"{'Variable':<28} {'KS stat':>8} {'p-value':>12} {'Mean clean':>12} {'Mean outlier':>13} {'Shift':>8}")
print("-" * 85)

ks_results = []
for j, var in enumerate(VAR_NAMES):
    v_clean = X_raw[mask_clean, j]
    v_outlier = X_raw[mask_outlier, j]
    stat, pval = ks_2samp(v_clean, v_outlier)
    shift = v_outlier.mean() - v_clean.mean()
    ks_results.append((var, stat, pval, v_clean.mean(), v_outlier.mean(), shift))
    flag = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
    print(f"{var:<28} {stat:>8.4f} {pval:>12.2e} {v_clean.mean():>12.4f} {v_outlier.mean():>13.4f} {shift:>+8.4f} {flag}")

print("\n*** p<0.001  ** p<0.01  * p<0.05")

In [ ]:
# ── Top variables les plus discriminantes (KS > seuil) ──
ks_sorted = sorted(ks_results, key=lambda x: x[1], reverse=True)
top_vars = [v for v in ks_sorted if v[2] < 0.01][:10]

if top_vars:
    fig, axes = plt.subplots(2, min(5, len(top_vars)), figsize=(18, 7))
    axes = axes.flat if len(top_vars) > 1 else [axes]

    for i, (var, stat, pval, _, _, _) in enumerate(top_vars[:min(10, len(axes))]):
        ax = axes[i]
        j = VAR_NAMES.index(var)
        ax.hist(X_raw[mask_clean, j], bins=40, alpha=0.6, label="Clean", density=True, color="steelblue")
        ax.hist(X_raw[mask_outlier, j], bins=40, alpha=0.6, label="Outlier", density=True, color="tomato")
        ax.set_title(f"{var}\nKS={stat:.3f}, p={pval:.1e}", fontsize=9, fontweight="bold")
        ax.legend(fontsize=7)

    for i in range(len(top_vars), len(axes)):
        axes[i].set_visible(False)

    fig.suptitle("Variables de design les plus discriminantes (clean vs outlier VLM)", fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("Aucune variable significativement discriminante (p < 0.01)")

## 4. Filtrage et constitution du dataset propre

On applique le filtre `_TARGET_CLIP` sur les 7 primitives simultanément :
un sample est conservé uniquement si **toutes** ses primitives sont dans les bornes physiques.

In [ ]:
# ── Application du filtre global ──
mask_all_clean = np.ones(len(df_Y), dtype=bool)
for key in PRIMITIVE_TARGETS:
    lo, hi = _TARGET_CLIP[key]
    vals = df_Y[key].values
    mask_all_clean &= (vals >= lo) & (vals <= hi)

n_clean = mask_all_clean.sum()
n_rejected = len(df_Y) - n_clean
print(f"Samples propres (toutes primitives in bounds) : {n_clean} / {len(df_Y)} ({100*n_clean/len(df_Y):.1f}%)")
print(f"Samples rejetés : {n_rejected} ({100*n_rejected/len(df_Y):.1f}%)")

X_clean = X_raw[mask_all_clean]
Y_clean = df_Y.values[mask_all_clean]
results_clean = [r for r, m in zip(results, mask_all_clean) if m]

df_X_clean = pd.DataFrame(X_clean, columns=VAR_NAMES)
df_Y_clean = pd.DataFrame(Y_clean, columns=PRIMITIVE_TARGETS)

print(f"\nDataset propre : X={X_clean.shape}, Y={Y_clean.shape}")

In [ ]:
# ── Distributions des primitives APRÈS filtrage ──
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flat

for i, key in enumerate(PRIMITIVE_TARGETS):
    ax = axes[i]
    vals = df_Y_clean[key].values
    lo, hi = _TARGET_CLIP[key]

    ax.hist(vals, bins=60, alpha=0.7, edgecolor="none", color="seagreen")
    ax.axvline(lo, color="red", ls="--", lw=1, alpha=0.5)
    ax.axvline(hi, color="red", ls="--", lw=1, alpha=0.5)

    ax.set_title(f"{key} (clean)\nmean={vals.mean():.4f}, std={vals.std():.4f}", fontsize=9, fontweight="bold")
    ax.set_ylabel("Count")

axes[7].set_visible(False)
fig.suptitle("Distribution des primitives après filtrage", fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Détection d'outliers résiduels par IQR sur le dataset propre ──
print("── Outliers résiduels (IQR × 3) sur dataset propre ──\n")
print(f"{'Cible':<12} {'Q1':>10} {'Q3':>10} {'IQR':>10} {'N mild (1.5×)':>14} {'N extrême (3×)':>15}")
print("-" * 75)

for key in PRIMITIVE_TARGETS:
    vals = df_Y_clean[key].values
    q1, q3 = np.percentile(vals, [25, 75])
    iqr = q3 - q1
    mild = ((vals < q1 - 1.5 * iqr) | (vals > q3 + 1.5 * iqr)).sum()
    extreme = ((vals < q1 - 3 * iqr) | (vals > q3 + 3 * iqr)).sum()
    print(f"{key:<12} {q1:>10.4f} {q3:>10.4f} {iqr:>10.4f} {mild:>14} {extreme:>15}")

## 5. Feature augmentation & analyse des corrélations

In [ ]:
# ── Feature augmentation (30 → 52) ──
X_aug = augment_features(X_clean)
print(f"Features augmentées : {X_clean.shape[1]} → {X_aug.shape[1]}")

AUG_NAMES = VAR_NAMES + [
    "wing_area", "aspect_ratio", "mac", "tip_chord", "body_root_chord",
    "outer_span", "total_twist", "twist_gradient", "mean_twist",
    "effective_span", "qc_sweep", "reflex_root", "camber_root",
    "ac_offset", "taper_elliptic_dev", "body_volume", "body_frontal_area",
    "body_wetted_area", "body_aspect_ratio", "blend_tc_mismatch",
    "body_sweep_total", "span_body_ratio",
]

df_aug = pd.DataFrame(X_aug, columns=AUG_NAMES)

# Vérification : pas de NaN ou Inf dans les features augmentées
n_nan = np.isnan(X_aug).sum()
n_inf = np.isinf(X_aug).sum()
print(f"NaN dans features augmentées : {n_nan}")
print(f"Inf dans features augmentées : {n_inf}")

df_aug.describe().round(4)

In [ ]:
# ── Corrélation features augmentées ↔ primitives VLM ──
corr_matrix = np.zeros((len(AUG_NAMES), len(PRIMITIVE_TARGETS)))
for i, fname in enumerate(AUG_NAMES):
    for j, tname in enumerate(PRIMITIVE_TARGETS):
        corr_matrix[i, j] = np.corrcoef(X_aug[:, i], Y_clean[:, j])[0, 1]

# Top 15 features les plus corrélées par primitive
fig, ax = plt.subplots(figsize=(10, 16))
sns.heatmap(corr_matrix, xticklabels=PRIMITIVE_TARGETS, yticklabels=AUG_NAMES,
            cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax, annot=False)
ax.set_title("Corrélation linéaire : features (52) vs primitives VLM (7)", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Top 5 features les plus corrélées pour chaque primitive ──
print("── Top 5 features par primitive (|corrélation| max) ──\n")
for j, tname in enumerate(PRIMITIVE_TARGETS):
    corrs = corr_matrix[:, j]
    top_idx = np.argsort(np.abs(corrs))[::-1][:5]
    print(f"{tname}:")
    for idx in top_idx:
        print(f"  {AUG_NAMES[idx]:<28} r = {corrs[idx]:+.4f}")
    print()

In [ ]:
# ── Corrélation inter-features (détection de multicolinéarité) ──
corr_feat = df_aug.corr()

# Paires à forte colinéarité (|r| > 0.95)
high_corr_pairs = []
for i in range(len(AUG_NAMES)):
    for j in range(i + 1, len(AUG_NAMES)):
        r = corr_feat.iloc[i, j]
        if abs(r) > 0.95:
            high_corr_pairs.append((AUG_NAMES[i], AUG_NAMES[j], r))

high_corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
print(f"Paires de features avec |r| > 0.95 : {len(high_corr_pairs)}\n")
for f1, f2, r in high_corr_pairs:
    print(f"  {f1:<28} ↔ {f2:<28} r = {r:+.4f}")

if not high_corr_pairs:
    print("  Aucune paire fortement colinéaire.")

## 6. Couverture du design space

Vérifie que le dataset propre couvre bien l'espace de design (pas de trous après filtrage)
et que le LHS n'est pas biaisé.

In [ ]:
# ── Couverture des bornes : ratio (range observé) / (range théorique) ──
lb = np.array([BOUNDS[v][0] for v in VAR_NAMES])
ub = np.array([BOUNDS[v][1] for v in VAR_NAMES])
design_range = ub - lb

print(f"{'Variable':<28} {'Borne lo':>9} {'Borne hi':>9} {'Min clean':>10} {'Max clean':>10} {'Couverture':>11}")
print("-" * 80)

coverage_pcts = []
for j, var in enumerate(VAR_NAMES):
    v = X_clean[:, j]
    obs_range = v.max() - v.min()
    coverage = obs_range / design_range[j] if design_range[j] > 0 else 1.0
    coverage_pcts.append(coverage)
    flag = " ⚠" if coverage < 0.80 else ""
    print(f"{var:<28} {lb[j]:>9.3f} {ub[j]:>9.3f} {v.min():>10.4f} {v.max():>10.4f} {100*coverage:>10.1f}%{flag}")

print(f"\nCouverture moyenne : {100*np.mean(coverage_pcts):.1f}%")
n_poor = sum(1 for c in coverage_pcts if c < 0.80)
if n_poor:
    print(f"⚠ {n_poor} variables avec couverture < 80%")

In [ ]:
# ── Uniformité : histogrammes normalisés des variables de design (clean) ──
from sklearn.preprocessing import MinMaxScaler

X_norm = MinMaxScaler().fit_transform(X_clean)

fig, axes = plt.subplots(5, 6, figsize=(20, 16))
axes = axes.flat

for j in range(N_VARS):
    ax = axes[j]
    ax.hist(X_norm[:, j], bins=30, alpha=0.7, edgecolor="none", color="teal", density=True)
    ax.axhline(1.0, color="red", ls="--", lw=0.8, alpha=0.6, label="Uniforme idéal")
    ax.set_title(VAR_NAMES[j], fontsize=8, fontweight="bold")
    ax.set_xlim(0, 1)
    ax.set_yticks([])

for j in range(N_VARS, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Uniformité des variables de design (normalisées [0,1]) — dataset propre\n"
             "Ligne rouge = distribution uniforme idéale", fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Biais de filtrage : quelles zones du design space sont sur-représentées/sous-représentées ? ──
# Compare la densité marginale avant/après filtrage pour chaque variable
fig, axes = plt.subplots(5, 6, figsize=(20, 16))
axes = axes.flat

X_norm_all = MinMaxScaler().fit_transform(X_raw)
X_norm_clean = MinMaxScaler().fit_transform(X_clean)

for j in range(N_VARS):
    ax = axes[j]
    ax.hist(X_norm_all[:, j], bins=30, alpha=0.4, density=True, color="gray", label="Brut")
    ax.hist(X_norm_clean[:, j], bins=30, alpha=0.6, density=True, color="steelblue", label="Propre")
    ax.set_title(VAR_NAMES[j], fontsize=8, fontweight="bold")
    ax.set_xlim(0, 1)
    ax.set_yticks([])
    if j == 0:
        ax.legend(fontsize=7)

for j in range(N_VARS, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Biais de filtrage : distribution brute (gris) vs propre (bleu)", fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 7. Analyse de la normalisation (StandardScaler)

Vérifie les propriétés statistiques des données normalisées telles que vues par le MLP.

In [ ]:
# ── StandardScaler sur X augmenté et Y ──
from sklearn.preprocessing import StandardScaler

scaler_X = StandardScaler().fit(X_aug)
scaler_Y = StandardScaler().fit(Y_clean)

X_scaled = scaler_X.transform(X_aug)
Y_scaled = scaler_Y.transform(Y_clean)

print("── Features (X) après StandardScaler ──")
print(f"{'Feature':<28} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10} {'Skew':>10} {'Kurt':>10}")
print("-" * 90)
from scipy.stats import skew, kurtosis
for j in range(X_scaled.shape[1]):
    col = X_scaled[:, j]
    name = AUG_NAMES[j] if j < len(AUG_NAMES) else f"feat_{j}"
    sk = skew(col)
    ku = kurtosis(col)
    flag = " ⚠" if abs(sk) > 2 or ku > 7 else ""
    print(f"{name:<28} {col.mean():>10.4f} {col.std():>10.4f} {col.min():>10.2f} {col.max():>10.2f} "
          f"{sk:>10.2f} {ku:>10.2f}{flag}")

print(f"\n── Targets (Y) après StandardScaler ──")
print(f"{'Target':<12} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10} {'Skew':>10} {'Kurt':>10}")
print("-" * 75)
for j, key in enumerate(PRIMITIVE_TARGETS):
    col = Y_scaled[:, j]
    sk = skew(col)
    ku = kurtosis(col)
    flag = " ⚠" if abs(sk) > 2 or ku > 7 else ""
    print(f"{key:<12} {col.mean():>10.4f} {col.std():>10.4f} {col.min():>10.2f} {col.max():>10.2f} "
          f"{sk:>10.2f} {ku:>10.2f}{flag}")

print("\n⚠ = skewness > 2 ou kurtosis > 7 (distribution non-gaussienne, potentiellement problématique pour le MLP)")

In [ ]:
# ── Boxplots des features normalisées (détection visuelle d'asymétrie) ──
fig, ax = plt.subplots(figsize=(20, 6))
bp = ax.boxplot(X_scaled, vert=True, patch_artist=True, widths=0.6,
                boxprops=dict(facecolor="lightblue", alpha=0.7),
                medianprops=dict(color="red", linewidth=1.5),
                flierprops=dict(marker='.', markersize=2, alpha=0.3))

ax.set_xticks(range(1, len(AUG_NAMES) + 1))
ax.set_xticklabels(AUG_NAMES, rotation=90, fontsize=7)
ax.set_ylabel("Valeur StandardScaler")
ax.set_title("Boxplots des 52 features (StandardScaler) — dataset propre", fontweight="bold")
ax.axhline(0, color="gray", ls="--", lw=0.5)
plt.tight_layout()
plt.show()

## 8. Corrélation inter-targets et structure des primitives

In [ ]:
# ── Corrélation entre les 7 primitives ──
corr_Y = df_Y_clean.corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_Y, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            ax=ax, square=True, linewidths=0.5)
ax.set_title("Corrélation entre les 7 primitives VLM (dataset propre)", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── Scatter matrix des primitives (pairplot) ──
g = sns.pairplot(df_Y_clean, diag_kind="kde", plot_kws={"s": 3, "alpha": 0.15},
                 diag_kws={"fill": True}, height=1.8)
g.figure.suptitle("Pairplot des 7 primitives VLM (dataset propre)", fontweight="bold", y=1.01)
plt.show()

## 9. Résumé & recommandations

In [ ]:
# ── Résumé automatique de la qualité des données ──
print("=" * 70)
print("         RÉSUMÉ — QUALITÉ DU DATASET POUR LE SURROGATE")
print("=" * 70)

print(f"\n1. TAILLE")
print(f"   Brut : {len(X_raw)} samples")
print(f"   Propre (toutes primitives in bounds) : {n_clean} ({100*n_clean/len(X_raw):.1f}%)")
print(f"   Rejetés (divergence VLM) : {n_rejected} ({100*n_rejected/len(X_raw):.1f}%)")

print(f"\n2. OUTLIERS VLM PAR PRIMITIVE")
for key in PRIMITIVE_TARGETS:
    lo, hi = _TARGET_CLIP[key]
    n_out = outlier_masks[key].sum()
    pct = 100 * n_out / len(df_Y)
    status = "OK" if pct < 5 else "ATTENTION" if pct < 20 else "CRITIQUE"
    print(f"   {key:<12} {n_out:>5} outliers ({pct:>5.1f}%) [{status}]")

print(f"\n3. FEATURES")
print(f"   Variables de design : {N_VARS}")
print(f"   Features augmentées : {X_aug.shape[1]}")
print(f"   NaN : {n_nan} | Inf : {n_inf}")
n_colinear = len(high_corr_pairs)
print(f"   Paires colinéaires (|r|>0.95) : {n_colinear}")

print(f"\n4. COUVERTURE DU DESIGN SPACE")
poor_coverage = [(VAR_NAMES[j], coverage_pcts[j]) for j in range(N_VARS) if coverage_pcts[j] < 0.80]
if poor_coverage:
    for var, cov in poor_coverage:
        print(f"   ⚠ {var}: {100*cov:.1f}% (sous-couverture)")
else:
    print(f"   Toutes les variables couvrent > 80% de leur plage")
print(f"   Couverture moyenne : {100*np.mean(coverage_pcts):.1f}%")

print(f"\n5. RECOMMANDATIONS POUR LE TRAINING")
print(f"   → Entraîner sur les {n_clean} samples propres (pas sur les 10k bruts)")
print(f"   → Évaluer le R² uniquement sur les samples propres")
if n_colinear > 3:
    print(f"   → {n_colinear} paires colinéaires : considérer PCA ou feature selection")
sk_targets = [(key, abs(skew(Y_scaled[:, j]))) for j, key in enumerate(PRIMITIVE_TARGETS)]
sk_high = [(k, s) for k, s in sk_targets if s > 1.5]
if sk_high:
    names = ", ".join(k for k, _ in sk_high)
    print(f"   → Targets asymétriques ({names}) : considérer log-transform ou robust scaler")

print("\n" + "=" * 70)